In [1]:
import pandas as pd

import seaborn as sns
import matplotlib.pyplot as plt

In [2]:
olist_path = "../Data/olist_post_cleaning.csv"

In [3]:
olist_df = pd.read_csv(olist_path)

olist_df

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,order_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,...,payment_installments,payment_value,review_score,seller_zip_code_prefix,seller_city,seller_state,product_category_name_english,freight_ratio,dates_diff,product_cat
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP,00e7ee1b050b8499577073aeb2a297a1,delivered,2017-05-16,2017-05-16,2017-05-23,...,2.0,146.87,4.0,8577.0,itaquaquecetuba,SP,office_furniture,0.175054,-11.0,office
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP,29150127e6685892b6eab3eec79f59c7,delivered,2018-01-12,2018-01-12,2018-01-15,...,8.0,335.48,5.0,88303.0,itajai,SC,housewares,0.160830,-8.0,household
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP,b2059ed67ce144a36e2aa97d2c9e9ad2,delivered,2018-05-19,2018-05-20,2018-06-11,...,7.0,157.73,5.0,8577.0,itaquaquecetuba,SP,office_furniture,0.127126,1.0,office
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP,951670f92359f4fe4a63112aa7306eba,delivered,2018-03-13,2018-03-13,2018-03-27,...,1.0,173.30,5.0,8577.0,itaquaquecetuba,SP,office_furniture,0.155796,-13.0,office
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP,6b7d50bd145f6fc7f33cebabd7e49d0f,delivered,2018-07-29,2018-07-29,2018-07-30,...,8.0,252.25,5.0,14940.0,ibitinga,SP,home_confort,0.096739,-6.0,household
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
118287,17ddf5dd5d51696bb3d7c6291687be6f,1a29b476fee25c95fbafc67c5ac95cf8,3937,sao paulo,SP,6760e20addcf0121e9d58f2f1ff14298,delivered,2018-04-07,2018-04-07,2018-04-11,...,6.0,88.78,4.0,17400.0,garca,SP,books_general_interest,0.185314,-12.0,books_media
118288,e7b71a9017aa05c9a7fd292d714858e8,d52a67c98be1cf6a5c84435bd38d095d,6764,taboao da serra,SP,9ec0c8947d973db4f4e8dcf1fbfa8f1b,delivered,2018-04-04,2018-04-04,2018-04-05,...,3.0,129.06,5.0,14802.0,araraquara,SP,sports_leisure,0.123238,-9.0,sports_leisure
118289,5e28dfe12db7fb50a4b2f691faecea5e,e9f50caf99f032f0bf3c55141f019d99,60115,fortaleza,CE,fed4434add09a6f332ea398efd656a5c,delivered,2018-04-08,2018-04-08,2018-04-09,...,5.0,56.04,1.0,3304.0,sao paulo,SP,health_beauty,0.514595,7.0,beauty
118290,56b18e2166679b8a959d72dd06da27f9,73c2643a0a458b49f58cea58833b192e,92120,canoas,RS,e31ec91cea1ecf97797787471f98a8c2,delivered,2017-11-03,2017-11-03,2017-11-06,...,2.0,711.07,5.0,14840.0,guariba,SP,watches_gifts,0.032032,-19.0,beauty


In [4]:
olist_df["order_purchase_timestamp"] = pd.to_datetime(olist_df["order_purchase_timestamp"])

-------
-------

# Training data set by days

## 90 days

### 90 days term 1 

In [5]:
pd.to_datetime("2016-12-01")

Timestamp('2016-12-01 00:00:00')

In [6]:
CHURN_DAYS = 90

In [7]:
start_date_90 = pd.to_datetime("2016-12-01")

start_date_90

Timestamp('2016-12-01 00:00:00')

In [8]:
ninety_days_later = start_date_90 + pd.Timedelta(days= CHURN_DAYS)

ninety_days_later

Timestamp('2017-03-01 00:00:00')

In [9]:
# Suming 90 days more to the variable ninety days later

end_date = ninety_days_later + pd.Timedelta(days= CHURN_DAYS)

end_date

Timestamp('2017-05-30 00:00:00')

In [10]:
is_customers = olist_df[
    (olist_df["order_purchase_timestamp"] >= start_date_90) & 
    (olist_df["order_purchase_timestamp"] < ninety_days_later)
]["customer_unique_id"].unique()

In [11]:
future_active_ids = set(
    olist_df[
        (olist_df["order_purchase_timestamp"] >= ninety_days_later) & 
        (olist_df["order_purchase_timestamp"] < end_date)
    ]["customer_unique_id"].unique()
)

In [12]:
obs_df = olist_df[olist_df["customer_unique_id"].isin(is_customers)].copy()
obs_df = obs_df[
    (obs_df["order_purchase_timestamp"] >= start_date_90) & 
    (obs_df["order_purchase_timestamp"] < ninety_days_later)
]

In [13]:
dataset_90_df = obs_df.groupby('customer_unique_id').agg({
    'price': 'sum',            # Total spent on the order (47.40 + 39.90)
    'freight_ratio': 'mean',    # Total shipping cost
    'payment_value': 'sum',    # Total amount charged to credit card
    'dates_diff': 'mean',
    'review_score': 'mean',

    # Categorical and indicator columns to pull forward
    'customer_city': 'first', 
    'customer_state': 'first', 
    'order_status': 'first',
    'payment_sequential': 'max',
    'payment_type': 'first',
    'payment_installments': 'max',
    'product_cat': 'first'

}).reset_index()

dataset_90_df

,customer_unique_id,price,freight_ratio,payment_value,dates_diff,review_score,customer_city,customer_state,order_status,payment_sequential,payment_type,payment_installments,product_cat
0,00115fc7123b5310cf6d3a3aa932699e,59.99,0.268711,76.11,-33.0,4.0,brasilia,DF,delivered,1.0,credit_card,1.0,sports_leisure
1,002b4cd83fabaffaa475f78ea5ef3e08,49.90,0.290982,64.42,-28.0,2.0,sao sebastiao do paraiso,MG,delivered,1.0,credit_card,1.0,fashion
2,0087eede471173af78d789df249c3a45,126.99,0.189936,151.11,-19.0,5.0,porto alegre,RS,delivered,1.0,boleto,1.0,miscellaneous
3,008f3d5f45a11059239a5c452cd00006,309.90,0.086028,336.56,-28.0,5.0,sao pedro da agua branca,MA,delivered,1.0,boleto,1.0,beauty
4,012b8001e47392df808a454083a74b74,65.60,0.497793,197.68,-28.0,5.0,capitao eneas,MG,delivered,1.0,credit_card,1.0,sports_leisure
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2441,ffa0ba4c9a6a0763879efe7c2b8d5b93,12.00,1.176667,26.12,0.0,1.0,sao paulo,SP,canceled,1.0,credit_card,1.0,household
2442,ffa46fd1f769dfbdd6c039550b420949,79.90,0.197372,95.67,-3.0,5.0,florianopolis,SC,delivered,1.0,credit_card,1.0,entertainment
2443,ffba9f9dff87b05e310ecc46c8591044,1591.20,0.022392,1626.83,-23.0,5.0,tailandia,PA,delivered,1.0,credit_card,5.0,beauty
2444,ffebb6424578e7bb153322da9d65634f,629.00,0.058347,665.70,-5.0,1.0,guarulhos,SP,delivered,1.0,credit_card,8.0,entertainment


In [14]:
dataset_90_df["churn_90"] =  (dataset_90_df["customer_unique_id"].isin(future_active_ids)).astype(int)

dataset_90_df["churn_90"]

0       0
1       0
2       0
3       0
4       0
       ..
2441    0
2442    0
2443    0
2444    0
2445    0
Name: churn_90, Length: 2446, dtype: int64

In [15]:
dataset_90_df["churn_90"].value_counts()

churn_90
0    2430
1      16
Name: count, dtype: int64

In [16]:
dataset_90_df.to_csv(r"C:\Users\Hallen\Documents\mis_cosas\silvana\master\tfm-uned-churn\Data\churn_dataset_90_1.csv", index=False)

------------------------------------ 

### **90 days term 2**

In [17]:
start_date_90_t2 = pd.to_datetime(end_date)

start_date_90_t2

Timestamp('2017-05-30 00:00:00')

In [18]:
ninety_days_later_t2 = start_date_90_t2 + pd.Timedelta(days= CHURN_DAYS)

ninety_days_later_t2

Timestamp('2017-08-28 00:00:00')

In [19]:
end_date_t2 = ninety_days_later_t2 + pd.Timedelta(days= CHURN_DAYS)

end_date_t2

Timestamp('2017-11-26 00:00:00')

In [20]:
is_customers_t2 = olist_df[
    (olist_df["order_purchase_timestamp"] >= start_date_90_t2) & 
    (olist_df["order_purchase_timestamp"] < ninety_days_later_t2)
]["customer_unique_id"].unique()

In [21]:
future_active_ids = set(
    olist_df[
        (olist_df["order_purchase_timestamp"] >= ninety_days_later_t2) & 
        (olist_df["order_purchase_timestamp"] < end_date_t2)
    ]["customer_unique_id"].unique()
)

In [22]:
obs_t2_df = olist_df[olist_df["customer_unique_id"].isin(is_customers_t2)].copy()
obs_t2_df = obs_t2_df[
    (obs_t2_df["order_purchase_timestamp"] >= start_date_90_t2) & 
    (obs_t2_df["order_purchase_timestamp"] < ninety_days_later_t2)]

In [23]:
dataset_90_t2_df = obs_t2_df.groupby('customer_unique_id').agg({
    'price': 'sum',            # Total spent on the order (47.40 + 39.90)
    'freight_ratio': 'mean',    # Total shipping cost
    'payment_value': 'sum',    # Total amount charged to credit card
    'dates_diff': 'mean',
    'review_score': 'mean',

    # Categorical and indicator columns to pull forward
    'customer_city': 'first', 
    'customer_state': 'first', 
    'order_status': 'first',
    'payment_sequential': 'max',
    'payment_type': 'first',
    'payment_installments': 'max',
    'product_cat': 'first'

}).reset_index()

dataset_90_t2_df

,customer_unique_id,price,freight_ratio,payment_value,dates_diff,review_score,customer_city,customer_state,order_status,payment_sequential,payment_type,payment_installments,product_cat
0,0006fdc98a402fceb4eb0ee528f6a8d4,13.90,1.086331,29.00,-12.0,3.0,mimoso do sul,ES,delivered,1.0,credit_card,2.0,household
1,000a5ad9c4601d2bbdd9ed765d5213b3,76.99,0.185609,91.28,-15.0,4.0,porto alegre,RS,delivered,1.0,credit_card,3.0,beauty
2,000de6019bb59f34c099a907c151d855,229.80,0.119279,514.88,-16.0,2.0,sao sebastiao,SP,delivered,1.0,credit_card,4.0,household
3,0010a452c6d13139e50b57f19f52e04e,299.00,0.090067,325.93,-19.0,1.0,taquara,RS,delivered,1.0,credit_card,10.0,household
4,0011857aff0e5871ce5eb429f21cdaf5,174.33,0.106121,192.83,-20.0,5.0,santo andre,SP,delivered,1.0,credit_card,3.0,fashion
...,...,...,...,...,...,...,...,...,...,...,...,...,...
10830,ffe780a8995715d9560ca10f3351710f,89.70,0.588629,427.50,-15.0,5.0,rio de janeiro,RJ,delivered,1.0,boleto,1.0,office
10831,ffe8f2fc0cee48f79934bd2c506fafc0,89.99,0.238138,111.42,-14.0,3.0,sao paulo,SP,delivered,1.0,boleto,1.0,electronics
10832,ffe9102bb78a76921ba0ff3c4659616a,385.26,0.080725,416.36,-21.0,4.0,rio de janeiro,RJ,delivered,1.0,debit_card,1.0,electronics
10833,fff3a9369e4b7102fab406a334a678c3,84.90,0.210130,102.74,-7.0,5.0,brasilia,DF,delivered,1.0,credit_card,2.0,fashion


In [24]:
dataset_90_t2_df["churn_90"] =  (dataset_90_t2_df["customer_unique_id"].isin(future_active_ids)).astype(int)

dataset_90_t2_df["churn_90"]

0        0
1        0
2        0
3        0
4        0
        ..
10830    0
10831    0
10832    0
10833    0
10834    0
Name: churn_90, Length: 10835, dtype: int64

In [25]:
dataset_90_t2_df["churn_90"].value_counts()

churn_90
0    10714
1      121
Name: count, dtype: int64

In [26]:
dataset_90_t2_df.to_csv(r"C:\Users\Hallen\Documents\mis_cosas\silvana\master\tfm-uned-churn\Data\churn_dataset_90_days2.csv", index=False)

-----

### **90 days term 3**

In [27]:
start_date_90_t3 = pd.to_datetime(end_date_t2)

start_date_90_t3

Timestamp('2017-11-26 00:00:00')

In [28]:
ninety_days_later_t3 = start_date_90_t3 + pd.Timedelta(days= CHURN_DAYS)

ninety_days_later_t3

Timestamp('2018-02-24 00:00:00')

In [29]:
end_date_t3 = ninety_days_later_t3 + pd.Timedelta(days= CHURN_DAYS)

end_date_t3

Timestamp('2018-05-25 00:00:00')

In [30]:
is_customers_t3 = olist_df[
    (olist_df["order_purchase_timestamp"] >= start_date_90_t3) & 
    (olist_df["order_purchase_timestamp"] < ninety_days_later_t3)
]["customer_unique_id"].unique()

In [31]:
future_active_ids = set(
    olist_df[
        (olist_df["order_purchase_timestamp"] >= ninety_days_later_t3) & 
        (olist_df["order_purchase_timestamp"] < end_date_t3)
    ]["customer_unique_id"].unique()
)

In [32]:
obs_t3_df = olist_df[olist_df["customer_unique_id"].isin(is_customers_t3)].copy()
obs_t3_df = obs_t3_df[
    (obs_t3_df["order_purchase_timestamp"] >= start_date_90_t3) & 
    (obs_t3_df["order_purchase_timestamp"] < ninety_days_later_t3)]


In [33]:
dataset_90_t3_df = obs_t3_df.groupby('customer_unique_id').agg({
    'price': 'sum',            # Total spent on the order (47.40 + 39.90)
    'freight_ratio': 'mean',    # Total shipping cost
    'payment_value': 'sum',    # Total amount charged to credit card
    'dates_diff': 'mean',
    'review_score': 'mean',

    # Categorical and indicator columns to pull forward
    'customer_city': 'first', 
    'customer_state': 'first', 
    'order_status': 'first',
    'payment_sequential': 'max',
    'payment_type': 'first',
    'payment_installments': 'max',
    'product_cat': 'first'

}).reset_index()

dataset_90_t3_df

,customer_unique_id,price,freight_ratio,payment_value,dates_diff,review_score,customer_city,customer_state,order_status,payment_sequential,payment_type,payment_installments,product_cat
0,000c8bdb58a29e7115cfc257230fb21b,13.90,1.086331,29.00,-9.0,5.0,belo horizonte,MG,delivered,1.0,credit_card,2.0,household
1,000d460961d6dbfa3ec6c9f5805769e1,28.90,0.269204,36.68,-13.0,5.0,sao paulo,SP,delivered,1.0,credit_card,1.0,electronics
2,0014a5a58da615f7b01a4f5e194bf5ea,88.00,0.134318,99.82,-7.0,5.0,sao paulo,SP,delivered,1.0,credit_card,3.0,beauty
3,00196fdb2bf9edfc35e88ebfbcf8d781,12.90,1.093023,27.00,1.0,3.0,canoas,RS,delivered,1.0,credit_card,1.0,electronics
4,0019da6aa6bcb27cc32f1249bd12da05,79.90,0.097747,87.71,-10.0,1.0,limeira,SP,delivered,1.0,credit_card,1.0,household
...,...,...,...,...,...,...,...,...,...,...,...,...,...
19569,ffe9e41fbd14db4a7361347c56af5447,199.00,0.109950,220.88,-9.0,5.0,vila velha,ES,delivered,1.0,credit_card,2.0,beauty
19570,fff7219c86179ca6441b8f37823ba3d3,245.80,0.081367,531.60,-12.0,4.0,cachoeiras de macacu,RJ,delivered,1.0,debit_card,1.0,household
19571,fffb09418989a0dbff854a28163e47c6,58.00,0.261379,73.16,-23.0,5.0,veranopolis,RS,delivered,1.0,boleto,1.0,beauty
19572,fffbf87b7a1a6fa8b03f081c5f51a201,149.00,0.122953,167.32,-14.0,5.0,fortaleza,CE,delivered,1.0,credit_card,2.0,unknown


In [34]:
dataset_90_t3_df["churn_90"] =  (dataset_90_t3_df["customer_unique_id"].isin(future_active_ids)).astype(int)

dataset_90_t3_df["churn_90"]

0        0
1        0
2        0
3        0
4        0
        ..
19569    0
19570    0
19571    0
19572    0
19573    0
Name: churn_90, Length: 19574, dtype: int64

In [35]:
dataset_90_t3_df["churn_90"].value_counts()

churn_90
0    19388
1      186
Name: count, dtype: int64

In [36]:
dataset_90_t3_df.to_csv(r"C:\Users\Hallen\Documents\mis_cosas\silvana\master\tfm-uned-churn\Data\churn_dataset_90_3.csv", index=False)

----------------------------------------------------------------------
----------------------------------

## 120 days

### **120 days term 1**

In [37]:
CHURN_DAYS = 120

In [38]:
start_date_120 = pd.to_datetime("2016-12-01")

start_date_120

Timestamp('2016-12-01 00:00:00')

In [39]:
one_hundred_twenty_days_later = start_date_120 + pd.Timedelta(days= CHURN_DAYS)

one_hundred_twenty_days_later

Timestamp('2017-03-31 00:00:00')

In [40]:
end_date_one_hundred_twenty = one_hundred_twenty_days_later + pd.Timedelta(days= CHURN_DAYS)

end_date_one_hundred_twenty

Timestamp('2017-07-29 00:00:00')

In [41]:
is_customers_one_hundred_twenty = olist_df[
    (olist_df["order_purchase_timestamp"] >= start_date_120) & 
    (olist_df["order_purchase_timestamp"] < one_hundred_twenty_days_later)
]["customer_unique_id"].unique()

In [42]:
future_active_ids = set(
    olist_df[
        (olist_df["order_purchase_timestamp"] >= one_hundred_twenty_days_later) & 
        (olist_df["order_purchase_timestamp"] < end_date_one_hundred_twenty)
    ]["customer_unique_id"].unique()
)

In [43]:
obs_one_hundred_twenty_df = olist_df[olist_df["customer_unique_id"].isin(is_customers_one_hundred_twenty)].copy()
obs_one_hundred_twenty_df = obs_one_hundred_twenty_df[
    (obs_one_hundred_twenty_df["order_purchase_timestamp"] >= start_date_120) & 
    (obs_one_hundred_twenty_df["order_purchase_timestamp"] < one_hundred_twenty_days_later)]

In [44]:
dataset_120_df = obs_one_hundred_twenty_df.groupby('customer_unique_id').agg({
    'price': 'sum',            # Total spent on the order (47.40 + 39.90)
    'freight_ratio': 'mean',    # Total shipping cost
    'payment_value': 'sum',    # Total amount charged to credit card
    'dates_diff': 'mean',
    'review_score': 'mean',

    # Categorical and indicator columns to pull forward
    'customer_city': 'first', 
    'customer_state': 'first', 
    'order_status': 'first',
    'payment_sequential': 'max',
    'payment_type': 'first',
    'payment_installments': 'max',
    'product_cat': 'first'

}).reset_index()

dataset_120_df

,customer_unique_id,price,freight_ratio,payment_value,dates_diff,review_score,customer_city,customer_state,order_status,payment_sequential,payment_type,payment_installments,product_cat
0,0000f46a3911fa3c0805444483337064,69.00,0.249565,86.22,-2.0,3.0,sao jose,SC,delivered,1.0,credit_card,8.0,office
1,0005e1862207bf6ccc02e4228effd9a0,135.00,0.112000,150.12,-28.0,4.0,teresopolis,RJ,delivered,1.0,credit_card,3.0,fashion
2,00115fc7123b5310cf6d3a3aa932699e,59.99,0.268711,76.11,-33.0,4.0,brasilia,DF,delivered,1.0,credit_card,1.0,sports_leisure
3,001f3c4211216384d5fe59b041ce1461,24.88,0.440514,35.84,-11.0,3.0,sao paulo,SP,delivered,1.0,credit_card,3.0,household
4,002043098f10ba39a4600b6c52fbfe3c,176.99,0.343635,237.81,-16.0,4.0,aracaju,SE,delivered,1.0,credit_card,4.0,office
...,...,...,...,...,...,...,...,...,...,...,...,...,...
4966,ffe2dd1f3b0cbf0b5f35e818ec03c49b,89.99,0.230137,110.70,-9.0,4.0,rio de janeiro,RJ,delivered,1.0,credit_card,2.0,household
4967,ffe3e199b9d0b7fb7d2d29a5b9498447,146.90,0.103472,162.10,0.0,1.0,uberlandia,MG,delivered,1.0,boleto,1.0,construction
4968,ffebb6424578e7bb153322da9d65634f,629.00,0.058347,665.70,-5.0,1.0,guarulhos,SP,delivered,1.0,credit_card,8.0,entertainment
4969,ffedff0547d809c90c05c2691c51f9b7,17.90,0.811173,32.42,-56.0,5.0,ribeirao preto,SP,delivered,1.0,credit_card,3.0,household


In [45]:
dataset_120_df["churn_120"] =  (dataset_120_df["customer_unique_id"].isin(future_active_ids)).astype(int)

dataset_120_df["churn_120"]

0       0
1       0
2       0
3       0
4       0
       ..
4966    0
4967    0
4968    0
4969    0
4970    0
Name: churn_120, Length: 4971, dtype: int64

In [46]:
dataset_120_df["churn_120"].value_counts()

churn_120
0    4912
1      59
Name: count, dtype: int64

In [47]:
dataset_120_df.to_csv(r"C:\Users\Hallen\Documents\mis_cosas\silvana\master\tfm-uned-churn\Data\churn_dataset_120_1.csv", index=False)

-------

### **120 days term 2**

In [48]:
start_date_120_t2 = end_date_one_hundred_twenty

start_date_120_t2

Timestamp('2017-07-29 00:00:00')

In [49]:
one_hundred_twenty_days_later_t2 = start_date_120_t2 + pd.Timedelta(days= CHURN_DAYS)

one_hundred_twenty_days_later_t2

Timestamp('2017-11-26 00:00:00')

In [50]:
end_date_one_hundred_twenty_t2 = one_hundred_twenty_days_later_t2 + pd.Timedelta(days= CHURN_DAYS)

end_date_one_hundred_twenty_t2

Timestamp('2018-03-26 00:00:00')

In [51]:
is_customers_one_hundred_twenty_t2 = olist_df[
    (olist_df["order_purchase_timestamp"] >= start_date_120_t2) & 
    (olist_df["order_purchase_timestamp"] < one_hundred_twenty_days_later_t2)
]["customer_unique_id"].unique()

In [52]:
future_active_ids = set(
    olist_df[
        (olist_df["order_purchase_timestamp"] >= one_hundred_twenty_days_later_t2) & 
        (olist_df["order_purchase_timestamp"] < end_date_one_hundred_twenty_t2)
    ]["customer_unique_id"].unique()
)

In [53]:
obs_one_hundred_twenty_t2df = olist_df[olist_df["customer_unique_id"].isin(is_customers_one_hundred_twenty_t2)].copy()
obs_one_hundred_twenty_t2df = obs_one_hundred_twenty_t2df[
    (obs_one_hundred_twenty_t2df["order_purchase_timestamp"] >= start_date_120_t2) & 
    (obs_one_hundred_twenty_t2df["order_purchase_timestamp"] < one_hundred_twenty_days_later_t2)]

In [54]:
dataset_120_t2df = obs_one_hundred_twenty_t2df.groupby('customer_unique_id').agg({
    'price': 'sum',            # Total spent on the order (47.40 + 39.90)
    'freight_ratio': 'mean',    # Total shipping cost
    'payment_value': 'sum',    # Total amount charged to credit card
    'dates_diff': 'mean',
    'review_score': 'mean',

    # Categorical and indicator columns to pull forward
    'customer_city': 'first', 
    'customer_state': 'first', 
    'order_status': 'first',
    'payment_sequential': 'max',
    'payment_type': 'first',
    'payment_installments': 'max',
    'product_cat': 'first'

}).reset_index()

dataset_120_t2df

,customer_unique_id,price,freight_ratio,payment_value,dates_diff,review_score,customer_city,customer_state,order_status,payment_sequential,payment_type,payment_installments,product_cat
0,0000f6ccb0745a6a4b88665a16c9f078,25.99,0.678338,43.62,-12.0,4.0,belem,PA,delivered,1.0,credit_card,4.0,electronics
1,0004aac84e0df4da2b147fca70cf8255,180.00,0.093833,196.89,-8.0,5.0,sorocaba,SP,delivered,1.0,credit_card,6.0,electronics
2,00082cbe03e478190aadbea78542e933,79.00,0.598228,126.26,-8.0,5.0,itapeva,SP,delivered,1.0,boleto,1.0,fashion
3,000a5ad9c4601d2bbdd9ed765d5213b3,76.99,0.185609,91.28,-15.0,4.0,porto alegre,RS,delivered,1.0,credit_card,3.0,beauty
4,000bfa1d2f1a41876493be685390d6d3,70.00,0.338571,93.70,-13.0,4.5,santos,SP,delivered,1.0,credit_card,4.0,fashion
...,...,...,...,...,...,...,...,...,...,...,...,...,...
18722,ffeefd086fc667aaf6595c8fe3d22d54,55.00,0.144364,62.94,-8.0,4.0,japeri,RJ,delivered,1.0,credit_card,3.0,beauty
18723,fff1afc79f6b5db1e235a4a6c30ceda7,34.99,0.431552,50.09,-16.0,5.0,erechim,RS,delivered,1.0,credit_card,4.0,beauty
18724,fff3a9369e4b7102fab406a334a678c3,84.90,0.210130,102.74,-7.0,5.0,brasilia,DF,delivered,1.0,credit_card,2.0,fashion
18725,fff699c184bcc967d62fa2c6171765f7,39.90,0.378446,55.00,-7.0,4.0,santos,SP,delivered,1.0,boleto,1.0,entertainment


In [55]:
dataset_120_t2df["churn_120"] =  (dataset_120_t2df["customer_unique_id"].isin(future_active_ids)).astype(int)

dataset_120_t2df["churn_120"]

0        0
1        0
2        0
3        0
4        0
        ..
18722    0
18723    0
18724    0
18725    0
18726    0
Name: churn_120, Length: 18727, dtype: int64

In [56]:
dataset_120_t2df["churn_120"].value_counts()

churn_120
0    18508
1      219
Name: count, dtype: int64

In [57]:
dataset_120_t2df.to_csv(r"C:\Users\Hallen\Documents\mis_cosas\silvana\master\tfm-uned-churn\Data\churn_dataset_120_2.csv", index=False)

-------
-------

## 150 days

### **150 days term 1**

In [58]:
CHURN_DAYS = 150

In [59]:
start_date_150 = pd.to_datetime("2016-12-01")

start_date_150

Timestamp('2016-12-01 00:00:00')

In [60]:
one_hundred_fifty_days_later = start_date_150 + pd.Timedelta(days= CHURN_DAYS)

one_hundred_fifty_days_later

Timestamp('2017-04-30 00:00:00')

In [61]:
end_date_hundred_fifty = one_hundred_fifty_days_later + pd.Timedelta(days= CHURN_DAYS)

end_date_hundred_fifty

Timestamp('2017-09-27 00:00:00')

In [62]:
is_customers_one_hundred_fifty = olist_df[
    (olist_df["order_purchase_timestamp"] >= start_date_150) & 
    (olist_df["order_purchase_timestamp"] < one_hundred_fifty_days_later)
]["customer_unique_id"].unique()

In [63]:
future_active_ids = set(
    olist_df[
        (olist_df["order_purchase_timestamp"] >= one_hundred_fifty_days_later) & 
        (olist_df["order_purchase_timestamp"] < end_date_hundred_fifty)
    ]["customer_unique_id"].unique()
)

In [64]:
obs_one_hundred_fifty_df = olist_df[olist_df["customer_unique_id"].isin(is_customers_one_hundred_fifty)].copy()
obs_one_hundred_fifty_df = obs_one_hundred_fifty_df[
    (obs_one_hundred_fifty_df["order_purchase_timestamp"] >= start_date_150) & 
    (obs_one_hundred_fifty_df["order_purchase_timestamp"] < one_hundred_fifty_days_later)]

In [65]:
dataset_150_df = obs_one_hundred_fifty_df.groupby('customer_unique_id').agg({
    'price': 'sum',            # Total spent on the order (47.40 + 39.90)
    'freight_ratio': 'mean',    # Total shipping cost
    'payment_value': 'sum',    # Total amount charged to credit card
    'dates_diff': 'mean',
    'review_score': 'mean',

    # Categorical and indicator columns to pull forward
    'customer_city': 'first', 
    'customer_state': 'first', 
    'order_status': 'first',
    'payment_sequential': 'max',
    'payment_type': 'first',
    'payment_installments': 'max',
    'product_cat': 'first'

}).reset_index()

dataset_150_df

,customer_unique_id,price,freight_ratio,payment_value,dates_diff,review_score,customer_city,customer_state,order_status,payment_sequential,payment_type,payment_installments,product_cat
0,0000f46a3911fa3c0805444483337064,69.00,0.249565,86.22,-2.0,3.0,sao jose,SC,delivered,1.0,credit_card,8.0,office
1,0005e1862207bf6ccc02e4228effd9a0,135.00,0.112000,150.12,-28.0,4.0,teresopolis,RJ,delivered,1.0,credit_card,3.0,fashion
2,00115fc7123b5310cf6d3a3aa932699e,59.99,0.268711,76.11,-33.0,4.0,brasilia,DF,delivered,1.0,credit_card,1.0,sports_leisure
3,0011805441c0d1b68b48002f1d005526,269.00,0.104610,297.14,-13.0,3.0,goianesia do para,PA,delivered,1.0,credit_card,10.0,electronics
4,00191a9719ef48ebb5860b130347bf33,47.90,0.228810,58.86,-19.0,3.0,jarinu,SP,delivered,1.0,credit_card,2.0,household
...,...,...,...,...,...,...,...,...,...,...,...,...,...
7308,ffe2dd1f3b0cbf0b5f35e818ec03c49b,89.99,0.230137,110.70,-9.0,4.0,rio de janeiro,RJ,delivered,1.0,credit_card,2.0,household
7309,ffe3e199b9d0b7fb7d2d29a5b9498447,146.90,0.103472,162.10,0.0,1.0,uberlandia,MG,delivered,1.0,boleto,1.0,construction
7310,ffebb6424578e7bb153322da9d65634f,629.00,0.058347,665.70,-5.0,1.0,guarulhos,SP,delivered,1.0,credit_card,8.0,entertainment
7311,ffedff0547d809c90c05c2691c51f9b7,17.90,0.811173,32.42,-56.0,5.0,ribeirao preto,SP,delivered,1.0,credit_card,3.0,household


In [66]:
dataset_150_df["churn_150"] =  (dataset_150_df["customer_unique_id"].isin(future_active_ids)).astype(int)

dataset_150_df["churn_150"]

0       0
1       0
2       0
3       0
4       0
       ..
7308    0
7309    0
7310    0
7311    0
7312    0
Name: churn_150, Length: 7313, dtype: int64

In [67]:
dataset_150_df["churn_150"].value_counts()

churn_150
0    7222
1      91
Name: count, dtype: int64

In [68]:
dataset_150_df.to_csv(r"C:\Users\Hallen\Documents\mis_cosas\silvana\master\tfm-uned-churn\Data\churn_dataset_150_1.csv", index=False)

-----

### **150 days term 2**

In [69]:
start_date_150_t2 = one_hundred_fifty_days_later

start_date_150_t2

Timestamp('2017-04-30 00:00:00')

In [70]:
one_hundred_fifty_days_later_t2 = start_date_150_t2 + pd.Timedelta(days= CHURN_DAYS)

one_hundred_fifty_days_later_t2

Timestamp('2017-09-27 00:00:00')

In [71]:
end_date_hundred_fifty_t2 = one_hundred_fifty_days_later_t2 + pd.Timedelta(days= CHURN_DAYS)

end_date_hundred_fifty_t2

Timestamp('2018-02-24 00:00:00')

In [72]:
is_customers_one_hundred_fifty_t2 = olist_df[
    (olist_df["order_purchase_timestamp"] >= start_date_150_t2) & 
    (olist_df["order_purchase_timestamp"] < one_hundred_fifty_days_later_t2)
]["customer_unique_id"].unique()

In [73]:
future_active_ids = set(
    olist_df[
        (olist_df["order_purchase_timestamp"] >= one_hundred_fifty_days_later_t2) & 
        (olist_df["order_purchase_timestamp"] < end_date_hundred_fifty_t2)
    ]["customer_unique_id"].unique()
)

In [74]:
obs_one_hundred_fifty_t2df = olist_df[olist_df["customer_unique_id"].isin(is_customers_one_hundred_fifty_t2)].copy()
obs_one_hundred_fifty_t2df = obs_one_hundred_fifty_t2df[
    (obs_one_hundred_fifty_t2df["order_purchase_timestamp"] >= start_date_150_t2) & 
    (obs_one_hundred_fifty_t2df["order_purchase_timestamp"] < one_hundred_fifty_days_later_t2)]

In [75]:
dataset_150_t2df = obs_one_hundred_fifty_t2df.groupby('customer_unique_id').agg({
    'price': 'sum',            # Total spent on the order (47.40 + 39.90)
    'freight_ratio': 'mean',    # Total shipping cost
    'payment_value': 'sum',    # Total amount charged to credit card
    'dates_diff': 'mean',
    'review_score': 'mean',

    # Categorical and indicator columns to pull forward
    'customer_city': 'first', 
    'customer_state': 'first', 
    'order_status': 'first',
    'payment_sequential': 'max',
    'payment_type': 'first',
    'payment_installments': 'max',
    'product_cat': 'first'

}).reset_index()

dataset_150_t2df

,customer_unique_id,price,freight_ratio,payment_value,dates_diff,review_score,customer_city,customer_state,order_status,payment_sequential,payment_type,payment_installments,product_cat
0,0006fdc98a402fceb4eb0ee528f6a8d4,13.90,1.086331,29.00,-12.0,3.0,mimoso do sul,ES,delivered,1.0,credit_card,2.0,household
1,000a5ad9c4601d2bbdd9ed765d5213b3,76.99,0.185609,91.28,-15.0,4.0,porto alegre,RS,delivered,1.0,credit_card,3.0,beauty
2,000de6019bb59f34c099a907c151d855,229.80,0.119279,514.88,-16.0,2.0,sao sebastiao,SP,delivered,1.0,credit_card,4.0,household
3,0010a452c6d13139e50b57f19f52e04e,299.00,0.090067,325.93,-19.0,1.0,taquara,RS,delivered,1.0,credit_card,10.0,household
4,001147e649a7b1afd577e873841632dd,170.00,0.248000,424.32,-14.0,4.0,maringa,PR,delivered,1.0,credit_card,1.0,household
...,...,...,...,...,...,...,...,...,...,...,...,...,...
18422,fff1afc79f6b5db1e235a4a6c30ceda7,34.99,0.431552,50.09,-16.0,5.0,erechim,RS,delivered,1.0,credit_card,4.0,beauty
18423,fff3a9369e4b7102fab406a334a678c3,84.90,0.210130,102.74,-7.0,5.0,brasilia,DF,delivered,1.0,credit_card,2.0,fashion
18424,fff699c184bcc967d62fa2c6171765f7,39.90,0.378446,55.00,-7.0,4.0,santos,SP,delivered,1.0,boleto,1.0,entertainment
18425,fffcf5a5ff07b0908bd4e2dbc735a684,1570.00,0.320939,4134.84,-27.0,5.0,sanharo,PE,delivered,1.0,credit_card,10.0,beauty


In [76]:
dataset_150_t2df["churn_150"] =  (dataset_150_t2df["customer_unique_id"].isin(future_active_ids)).astype(int)

dataset_150_t2df["churn_150"]

0        0
1        0
2        0
3        0
4        0
        ..
18422    0
18423    0
18424    0
18425    0
18426    0
Name: churn_150, Length: 18427, dtype: int64

In [77]:
dataset_150_t2df["churn_150"].value_counts()

churn_150
0    18140
1      287
Name: count, dtype: int64

In [78]:
dataset_150_t2df.to_csv(r"C:\Users\Hallen\Documents\mis_cosas\silvana\master\tfm-uned-churn\Data\churn_dataset_150_2.csv", index=False)

--------
--------

Calculating Max the test dates range

In [79]:
max_date = olist_df["order_purchase_timestamp"].max()

max_date

Timestamp('2018-09-03 00:00:00')

In [80]:
limit_test = max_date - (pd.Timedelta(days= 110)*2)

test_period = limit_test - pd.Timedelta(days= 10)

test_period

Timestamp('2018-01-16 00:00:00')

------
------

# Test data set by days

## 90 days

In [81]:
CHURN_DAYS = 90

In [82]:
start_date_90T = test_period

start_date_90T

Timestamp('2018-01-16 00:00:00')

In [83]:
ninety_days_laterT = start_date_90T + pd.Timedelta(days= CHURN_DAYS)

ninety_days_laterT

Timestamp('2018-04-16 00:00:00')

In [84]:
end_dateT = ninety_days_laterT + pd.Timedelta(days= CHURN_DAYS)

end_dateT

Timestamp('2018-07-15 00:00:00')

In [85]:
is_customers_90 = olist_df[
    (olist_df["order_purchase_timestamp"] >= start_date_90T) & 
    (olist_df["order_purchase_timestamp"] < ninety_days_laterT)
]["customer_unique_id"].unique()

In [86]:
future_active_ids = set(
    olist_df[
        (olist_df["order_purchase_timestamp"] >= ninety_days_laterT) & 
        (olist_df["order_purchase_timestamp"] < end_dateT)
    ]["customer_unique_id"].unique()
)

In [87]:
obs_df = olist_df[olist_df["customer_unique_id"].isin(is_customers_90)].copy()
obs_df = obs_df[
    (obs_df["order_purchase_timestamp"] >= start_date_90T) & 
    (obs_df["order_purchase_timestamp"] < ninety_days_laterT)]

In [88]:
dataset_90_test_df = obs_df.groupby('customer_unique_id').agg({
    'price': 'sum',            # Total spent on the order (47.40 + 39.90)
    'freight_ratio': 'mean',    # Total shipping cost
    'payment_value': 'sum',    # Total amount charged to credit card
    'dates_diff': 'mean',
    'review_score': 'mean',

    # Categorical and indicator columns to pull forward
    'customer_city': 'first', 
    'customer_state': 'first', 
    'order_status': 'first',
    'payment_sequential': 'max',
    'payment_type': 'first',
    'payment_installments': 'max',
    'product_cat': 'first'

}).reset_index()

dataset_90_test_df

,customer_unique_id,price,freight_ratio,payment_value,dates_diff,review_score,customer_city,customer_state,order_status,payment_sequential,payment_type,payment_installments,product_cat
0,0004bd2a26a76fe21f786e4fbd80607f,154.00,0.084286,166.98,-12.0,4.0,sao paulo,SP,delivered,1.0,credit_card,8.0,miscellaneous
1,00053a61a98854899e70ed204dd4bafe,382.00,0.097330,838.36,-10.0,1.0,curitiba,PR,delivered,1.0,credit_card,3.0,sports_leisure
2,0005ef4cd20d2893f0d9fbd94d3c0d97,104.90,0.236988,129.76,31.0,1.0,sao luis,MA,delivered,1.0,credit_card,4.0,sports_leisure
3,00090324bbad0e9342388303bb71ba0a,49.95,0.274474,63.66,-8.0,5.0,campinas,SP,delivered,1.0,credit_card,3.0,household
4,0010fb34b966d44409382af9e8fd5b77,49.95,0.237237,61.80,2.0,4.0,sao paulo,SP,delivered,1.0,credit_card,6.0,household
...,...,...,...,...,...,...,...,...,...,...,...,...,...
20563,ffe6efca3c7e6a06bad0a6a883280a93,213.90,0.060776,226.90,-9.0,4.0,praia grande,SP,delivered,1.0,credit_card,2.0,beauty
20564,ffe9e41fbd14db4a7361347c56af5447,199.00,0.109950,220.88,-9.0,5.0,vila velha,ES,delivered,1.0,credit_card,2.0,beauty
20565,ffec10ad4229ba46818560e1c8b40a68,120.00,0.131000,135.72,-10.0,5.0,viana,ES,delivered,1.0,credit_card,3.0,sports_leisure
20566,fff1bdd5c5e37ca79dd74deeb91aa5b6,134.90,0.282283,172.98,32.0,1.0,araguaina,TO,delivered,2.0,debit_card,1.0,entertainment


In [89]:
dataset_90_test_df["churn_90"] =  (dataset_90_test_df["customer_unique_id"].isin(future_active_ids)).astype(int)

dataset_90_test_df["churn_90"]

0        0
1        0
2        0
3        0
4        0
        ..
20563    0
20564    0
20565    0
20566    0
20567    0
Name: churn_90, Length: 20568, dtype: int64

In [90]:
dataset_90_test_df["churn_90"].value_counts()

churn_90
0    20385
1      183
Name: count, dtype: int64

In [91]:
dataset_90_test_df.to_csv(r"C:\Users\Hallen\Documents\mis_cosas\silvana\master\tfm-uned-churn\Data\churn_test_dataset_90.csv", index=False)

___

## 120 days

In [92]:
CHURN_DAYS = 120

In [93]:
one_hundred_twenty_days_laterT120 = start_date_90T + pd.Timedelta(days= CHURN_DAYS)

one_hundred_twenty_days_laterT120

Timestamp('2018-05-16 00:00:00')

In [94]:
end_dateT120 = one_hundred_twenty_days_laterT120 + pd.Timedelta(days= CHURN_DAYS)

end_dateT120

Timestamp('2018-09-13 00:00:00')

In [95]:
is_customers_120 = olist_df[
    (olist_df["order_purchase_timestamp"] >= start_date_90T) & 
    (olist_df["order_purchase_timestamp"] < one_hundred_twenty_days_laterT120)
]["customer_unique_id"].unique()

In [96]:
future_active_ids = set(
    olist_df[
        (olist_df["order_purchase_timestamp"] >= one_hundred_twenty_days_laterT120) & 
        (olist_df["order_purchase_timestamp"] < end_dateT120)
    ]["customer_unique_id"].unique())

In [97]:
obs_120_df = olist_df[olist_df["customer_unique_id"].isin(is_customers_120)].copy()
obs_120_df = obs_120_df[
    (obs_120_df["order_purchase_timestamp"] >= start_date_90T) & 
    (obs_120_df["order_purchase_timestamp"] < one_hundred_twenty_days_laterT120)]

In [98]:
dataset_120_test_df = obs_120_df.groupby('customer_unique_id').agg({
    'price': 'sum',            # Total spent on the order (47.40 + 39.90)
    'freight_ratio': 'mean',    # Total shipping cost
    'payment_value': 'sum',    # Total amount charged to credit card
    'dates_diff': 'mean',
    'review_score': 'mean',

    # Categorical and indicator columns to pull forward
    'customer_city': 'first', 
    'customer_state': 'first', 
    'order_status': 'first',
    'payment_sequential': 'max',
    'payment_type': 'first',
    'payment_installments': 'max',
    'product_cat': 'first'

}).reset_index()

dataset_120_test_df

,customer_unique_id,price,freight_ratio,payment_value,dates_diff,review_score,customer_city,customer_state,order_status,payment_sequential,payment_type,payment_installments,product_cat
0,0000366f3b9a7992bf8c76cfdf3221e2,129.90,0.092379,141.90,-5.0,5.0,cajamar,SP,delivered,1.0,credit_card,8.0,household
1,0000b849f77a49e4a4ce2b2a4ca5be3f,18.90,0.438624,27.19,-5.0,4.0,osasco,SP,delivered,1.0,credit_card,1.0,beauty
2,0004bd2a26a76fe21f786e4fbd80607f,154.00,0.084286,166.98,-12.0,4.0,sao paulo,SP,delivered,1.0,credit_card,8.0,miscellaneous
3,00050ab1314c0e55a6ca13cf7181fecf,27.99,0.264023,35.38,-12.0,4.0,campinas,SP,delivered,1.0,boleto,1.0,electronics
4,00053a61a98854899e70ed204dd4bafe,382.00,0.097330,838.36,-10.0,1.0,curitiba,PR,delivered,1.0,credit_card,3.0,sports_leisure
...,...,...,...,...,...,...,...,...,...,...,...,...,...
28212,ffeddf8aa7cdecf403e77b2e9a99e2ea,330.00,0.237576,204.20,6.0,2.0,santarem,PA,delivered,2.0,credit_card,5.0,office
28213,fff1bdd5c5e37ca79dd74deeb91aa5b6,134.90,0.282283,172.98,32.0,1.0,araguaina,TO,delivered,2.0,debit_card,1.0,entertainment
28214,fff2ae16b99c6f3c785f0e052f2a9cfb,129.94,0.546098,200.90,-21.0,5.0,rio de janeiro,RJ,delivered,1.0,credit_card,8.0,office
28215,fffcc512b7dfecaffd80f13614af1d16,688.00,0.032994,710.70,0.0,1.0,cabo frio,RJ,shipped,1.0,credit_card,3.0,beauty


In [99]:
dataset_120_test_df["churn_120"] =  (dataset_120_test_df["customer_unique_id"].isin(future_active_ids)).astype(int)

dataset_120_test_df["churn_120"]

0        0
1        0
2        0
3        0
4        0
        ..
28212    0
28213    0
28214    0
28215    0
28216    0
Name: churn_120, Length: 28217, dtype: int64

In [100]:
dataset_120_test_df["churn_120"].value_counts()

churn_120
0    27972
1      245
Name: count, dtype: int64

In [101]:
dataset_120_test_df.to_csv(r"C:\Users\Hallen\Documents\mis_cosas\silvana\master\tfm-uned-churn\Data\churn_test_dataset_120.csv", index=False)

-------

## 150 days

In [102]:
CHURN_DAYS = 150

In [103]:
one_hundred_fifty_days_laterT = start_date_90T + pd.Timedelta(days= CHURN_DAYS)

one_hundred_fifty_days_laterT

Timestamp('2018-06-15 00:00:00')

In [104]:
end_dateT150 = one_hundred_fifty_days_laterT + pd.Timedelta(days= CHURN_DAYS)

end_dateT150

Timestamp('2018-11-12 00:00:00')

In [105]:
is_customers_150 = olist_df[
    (olist_df["order_purchase_timestamp"] >= start_date_90T) & 
    (olist_df["order_purchase_timestamp"] < one_hundred_fifty_days_laterT)
]["customer_unique_id"].unique()

In [106]:
future_active_ids = set(
    olist_df[
        (olist_df["order_purchase_timestamp"] >= one_hundred_fifty_days_laterT) & 
        (olist_df["order_purchase_timestamp"] < end_dateT150)
    ]["customer_unique_id"].unique())

In [107]:
obs_150_df = olist_df[olist_df["customer_unique_id"].isin(is_customers_150)].copy()
obs_150_df = obs_150_df[
    (obs_150_df["order_purchase_timestamp"] >= start_date_90T) & 
    (obs_150_df["order_purchase_timestamp"] < one_hundred_fifty_days_laterT)]

In [108]:
dataset_150_test_df = obs_150_df.groupby('customer_unique_id').agg({
    'price': 'sum',            # Total spent on the order (47.40 + 39.90)
    'freight_ratio': 'mean',    # Total shipping cost
    'payment_value': 'sum',    # Total amount charged to credit card
    'dates_diff': 'mean',
    'review_score': 'mean',

    # Categorical and indicator columns to pull forward
    'customer_city': 'first', 
    'customer_state': 'first', 
    'order_status': 'first',
    'payment_sequential': 'max',
    'payment_type': 'first',
    'payment_installments': 'max',
    'product_cat': 'first'

}).reset_index()

dataset_150_test_df

,customer_unique_id,price,freight_ratio,payment_value,dates_diff,review_score,customer_city,customer_state,order_status,payment_sequential,payment_type,payment_installments,product_cat
0,0000366f3b9a7992bf8c76cfdf3221e2,129.90,0.092379,141.90,-5.0,5.0,cajamar,SP,delivered,1.0,credit_card,8.0,household
1,0000b849f77a49e4a4ce2b2a4ca5be3f,18.90,0.438624,27.19,-5.0,4.0,osasco,SP,delivered,1.0,credit_card,1.0,beauty
2,0004bd2a26a76fe21f786e4fbd80607f,154.00,0.084286,166.98,-12.0,4.0,sao paulo,SP,delivered,1.0,credit_card,8.0,miscellaneous
3,00050ab1314c0e55a6ca13cf7181fecf,27.99,0.264023,35.38,-12.0,4.0,campinas,SP,delivered,1.0,boleto,1.0,electronics
4,00053a61a98854899e70ed204dd4bafe,382.00,0.097330,838.36,-10.0,1.0,curitiba,PR,delivered,1.0,credit_card,3.0,sports_leisure
...,...,...,...,...,...,...,...,...,...,...,...,...,...
33672,ffeddf8aa7cdecf403e77b2e9a99e2ea,330.00,0.237576,204.20,6.0,2.0,santarem,PA,delivered,2.0,credit_card,5.0,office
33673,fff1bdd5c5e37ca79dd74deeb91aa5b6,134.90,0.282283,172.98,32.0,1.0,araguaina,TO,delivered,2.0,debit_card,1.0,entertainment
33674,fff2ae16b99c6f3c785f0e052f2a9cfb,129.94,0.546098,200.90,-21.0,5.0,rio de janeiro,RJ,delivered,1.0,credit_card,8.0,office
33675,fffcc512b7dfecaffd80f13614af1d16,688.00,0.032994,710.70,0.0,1.0,cabo frio,RJ,shipped,1.0,credit_card,3.0,beauty


In [109]:
dataset_150_test_df["churn_150"] =  (dataset_150_test_df["customer_unique_id"].isin(future_active_ids)).astype(int)

dataset_150_test_df["churn_150"]

0        0
1        0
2        0
3        0
4        0
        ..
33672    0
33673    0
33674    0
33675    0
33676    0
Name: churn_150, Length: 33677, dtype: int64

In [110]:
dataset_150_test_df["churn_150"].value_counts()

churn_150
0    33479
1      198
Name: count, dtype: int64

In [111]:
dataset_150_test_df.to_csv(r"C:\Users\Hallen\Documents\mis_cosas\silvana\master\tfm-uned-churn\Data\churn_test_dataset_150.csv", index=False)